In [1]:
import os
import re
import json
import time
import numpy as np
from langchain.chains.question_answering import load_qa_chain 
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chat_models import AzureChatOpenAI
from langchain.embeddings.openai import OpenAIEmbeddings 
from langchain.vectorstores.azuresearch import AzureSearch
from langchain.embeddings import OpenAIEmbeddings
from langchain.document_loaders import PDFPlumberLoader
from azure.search.documents import SearchClient  
from azure.core.credentials import AzureKeyCredential

import warnings
warnings.filterwarnings("ignore")

In [9]:
import mlflow
from langchain.chat_models import AzureChatOpenAI
from datetime import datetime
import uuid
import time
from typing import Dict, Any
from mlflow.tracking import MlflowClient
import json

class MLFlowFeedbackLogger:
    def __init__(self, azure_model, experiment_name="feedback_trial"):
        """
        Initialize the feedback logger with Azure OpenAI model and MLflow settings
        """
        self.azure_model = azure_model
        self.client = MlflowClient()
        self.experiment = mlflow.set_experiment(experiment_name)
        self.improved_responses = {}  # Store improved responses based on feedback

    def get_binary_feedback(self, response: str) -> int:
        """Get thumbs up/down feedback from user"""
        print("\nResponse:", response)
        print("\nWas this response helpful?")
        print("1: Thumbs Up")
        print("0: Thumbs Down")
        
        while True:
            try:
                feedback = input("Your feedback (1 or 0): ")
                if feedback in ['0', '1']:
                    return int(feedback)
                print("Please enter either 1 (thumbs up) or 0 (thumbs down)")
            except ValueError:
                print("Please enter a valid input (1 or 0)")

    def log_interaction(self, prompt: str, response: str, feedback: int, corrective_feedback: str = None):
        """
        Log a single interaction with binary feedback to MLflow
        """
        with mlflow.start_run():
            # Log model and response parameters
            mlflow.log_params({
                "model_name": self.azure_model.model_name,
                "deployment": self.azure_model.deployment_name,
                "temperature": self.azure_model.temperature,
                "max_tokens": self.azure_model.max_tokens,
            })

            # Log prompt, response, and feedback
            mlflow.log_text(prompt, "prompt.txt")
            mlflow.log_text(response, "response.txt")
            mlflow.log_metric("user_satisfaction", feedback)

            # Log response-related metrics
            response_metrics = {
                "response_length": len(response),
                "token_estimate": len(response.split()),
                "prompt_length": len(prompt)
            }
            mlflow.log_metrics(response_metrics)

            if corrective_feedback:
                mlflow.log_text(corrective_feedback, "feedback_response.txt")

            # Log additional tags
            mlflow.set_tags({
                "interaction_id": str(uuid.uuid4()),
                "timestamp": datetime.now().isoformat(),
                "feedback_type": "binary"
            })

    def log_improved_response(self, prompt: str, improved_response: str):
        """Log improved response when feedback is negative"""
        self.improved_responses[prompt] = improved_response

def get_model_response(azure_model, prompt: str) -> str:
    """Get response from Azure OpenAI model and log response time"""
    try:
        with mlflow.start_run(nested=True):
            start_time = time.time()
            response = azure_model.predict(prompt)
            end_time = time.time()

            # Log response time
            mlflow.log_metric("response_time", end_time - start_time)
            
            return response
    except Exception as e:
        mlflow.log_param("error", str(e))
        raise

In [ ]:

def main():
    azure_model = AzureChatOpenAI(
        openai_api_key="",
        openai_api_version="",
        openai_api_base="",
        deployment_name='',
        model_name="gpt-4-32k",
        temperature=0.1,
        max_tokens=2000
    )
    
    feedback_logger = MLFlowFeedbackLogger(azure_model)
    
    while True:
        # Get user input
        print("\nEnter your question (or 'quit' to exit):")
        prompt = input("> ")
        
        if prompt.lower() == 'quit':
            break

        if prompt in feedback_logger.improved_responses:
            print("\nDisplaying improved response:")
            response = feedback_logger.improved_responses[prompt]
        else:
            try:
                # Get model response
                print("\nGenerating response...")
                response = get_model_response(azure_model, prompt)
                
                # Get feedback from the user
                feedback = feedback_logger.get_binary_feedback(response)
                
                corrective_feedback = None
                # If feedback is negative, ask for corrective feedback and refine response
                if feedback == 0:
                    corrective_feedback = input("What could be improved in the response? ")
                    refined_prompt = prompt + " " + corrective_feedback  # Append corrective feedback to prompt
                    refined_response = get_model_response(azure_model, refined_prompt)  # Generate improved response
                    
                    # Log the improved response for future interactions
                    feedback_logger.log_improved_response(prompt, refined_response)
                    response = refined_response
                
                # Log interaction, including corrective feedback if provided
                feedback_logger.log_interaction(
                    prompt=prompt,
                    response=response,
                    feedback=feedback,
                    corrective_feedback=corrective_feedback
                )
                
                print("\nFeedback logged successfully!")
            
            except Exception as e:
                print(f"An error occurred: {str(e)}")
                continue

        # Display the final answer
        print("Final Answer:", response)

        # Display logged metrics for user inspection
        current_run = mlflow.active_run()
        if current_run:
            run_id = current_run.info.run_id
            metrics = mlflow.get_run(run_id).data.metrics
            print("\nMetrics logged to MLflow:")
            print("Run ID:", run_id)
            print("Metrics:", metrics)

if __name__ == "__main__":
    main()
